In [4]:
%cat /kaggle/input/datasets/amirint/pgsm8k/persian_gsm8k_cot_zeroshot.yaml

tag:
  - math_word_problems
task: persian_gsm8k_cot_zeroshot
dataset_path: AmirFazlollahi/pgsm8k
dataset_name: default
output_type: generate_until
training_split: train
fewshot_split: train
test_split: test
doc_to_text: "پرسش: {{question}}\nپاسخ: بیا گام به گام فکر کنیم."
doc_to_target: "{{answer}}" #" {{answer.split('### ')[-1].rstrip()}}"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_case: true
    ignore_punctuation: false
    regexes_to_ignore:
      - '،'
      - ","
      - "\\$"
      - "(?s).*#### "
      - "\\.$"
generation_kwargs:
  until:
    - "پرسش:"
    - "</s>"
    - "<|im_end|>"
  do_sample: false
repeats: 1
num_fewshot: 0
filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "پاسخ (\\-?[۰-۹\\.\\,،]+) است."
      - function: "take_first"
  - name: "flexible-extract"
    filter:
      - function: "regex"
        group_select: -1
        regex_pattern: "-?[۰-۹٠-٩0-9]+(?:[٫٬.,،]

In [5]:
!git clone https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
!pip install -e .
!pip install accelerate bitsandbytes

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 64058, done.
remote: Total 64058 (delta 0), reused 0 (delta 0), pack-reused 64058 (from 1)
Receiving objects: 100% (64058/64058), 34.64 MiB | 19.42 MiB/s, done.
Resolving deltas: 100% (44231/44231), done.
/kaggle/working/lm-evaluation-harness
Obtaining file:///kaggle/working/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 7.1 MB/s eta 0:00:00
  Building editable for lm_eval (pyproject.toml) ... done
  Created wheel for lm_eval: filename=lm_eval-0.4

In [3]:
!python -m lm_eval validate --tasks persian_gsm8k_cot_zeroshot --include_path /kaggle/input/datasets/amirint/pgsm8k


2026-06-05:08:35:06 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:08:35:06 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
Validating tasks: ['persian_gsm8k_cot_zeroshot']
All tasks found and valid


**persian gsm8k - base model**

In [11]:
!accelerate launch -m lm_eval --model hf \
    --model_args pretrained=unsloth/Qwen3-4B-Base \
    --tasks persian_gsm8k \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 1 \
    --limit 50 \
    --log_samples \
    --output_path /kaggle/working/results



The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-05:08:57:43 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:08:57:43 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:08:57:54 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:08:57:54 WARNING  [tasks._index:70] Task 'math_word_problems' from None ov

**persian gsm8k - fine-tunned model**

In [4]:
!accelerate launch -m lm_eval --model hf \
    --model_args pretrained=arefehRajabian/Qwen3-4B-Base-persian-math-grpo \
    --tasks persian_gsm8k \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 50 \
    --log_samples \
    --output_path /kaggle/working/results

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-05:08:37:06 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:08:37:06 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:08:37:18 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:08:37:18 WARNING  [tasks._index:70] Task 'math_word_problems' from None ov

**persian gsm8k - base model - persian_gsm8k_cot task**

In [7]:
!accelerate launch -m lm_eval --model hf \
    --model_args pretrained=unsloth/Qwen3-4B-Base \
    --tasks persian_gsm8k_cot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 50 \
    --log_samples \
    --output_path /content/persian-results3

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-05:10:54:27 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:10:54:27 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:10:54:38 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:10:54:38 WARNING  [tasks._index:70] Task 'math_word_problems' from None ov

**persian gsm8k - fine-tunned model - persian_gsm8k_cot task**

In [6]:
!accelerate launch -m lm_eval --model hf \
    --model_args pretrained=arefehRajabian/Qwen3-4B-Base-persian-math-grpo \
    --tasks persian_gsm8k_cot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 50 \
    --log_samples \
    --output_path /content/persian-results3

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-05:10:45:22 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:10:45:22 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:10:45:33 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:10:45:33 WARNING  [tasks._index:70] Task 'math_word_problems' from None ov

**persian gsm8k - base model - persian_gsm8k_cot_zeroshot task**

In [11]:
!accelerate launch -m lm_eval --model hf \
    --model_args pretrained=unsloth/Qwen3-4B-Base  \
    --tasks persian_gsm8k_cot_zeroshot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --gen_kwargs max_new_tokens=1024 \
    --device cuda:0 \
    --batch_size 4 \
    --limit 50 \
    --log_samples \
    --output_path /content/persian-results4


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-05:11:41:24 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:11:41:24 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:11:41:35 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:11:41:35 WARNING  [tasks._index:70] Task 'math_word_problems' from None ov

**persian gsm8k - fine-tune model - persian_gsm8k_cot_zeroshot task**

In [9]:
!accelerate launch -m lm_eval --model hf \
    --model_args pretrained=arefehRajabian/Qwen3-4B-Base-persian-math-grpo \
    --tasks persian_gsm8k_cot_zeroshot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --gen_kwargs max_new_tokens=1024 \
    --device cuda:0 \
    --batch_size 8 \
    --limit 50 \
    --log_samples \
    --output_path /content/persian-results4



The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-06-05:11:18:35 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:11:18:35 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-06-05:11:18:45 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-06-05:11:18:45 WARNING  [tasks._index:70] Task 'math_word_problems' from None ov